# On-distribution AWM eval — base vs finetuned (step 1)

Tests whether the async-GRPO finetune beats the base model **on the same AWM tasks it trained on**, scored with the deterministic code verifier. If finetuned does **not** beat base here, the training produced no usable signal (so the BFCL no-improvement is a training problem, not just a domain-transfer gap).

Runtime: **GPU** (T4 works; A100 faster). The AWM env runs on the hosted HF Space — no local env server. vLLM serves each model in turn through the same harness used in training (custom system prompt, native `list_tools`/`call_tool` tool-calling, Qwen3 thinking mode, temp 1.0).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Install vLLM, the OpenAI client, and the AWM env client (from the public OpenEnv fork).
!pip -q install vllm openai
!pip -q install "git+https://github.com/Jake-Song/OpenEnv.git#subdirectory=envs/agent_world_model_env"

# Get the eval script + task list.
![ -d research ] || git clone -q https://github.com/Jake-Song/research.git
%cd research
!git fetch -q origin && git checkout -q exp/awm-dapo && git pull -q
!ls experiment/awm_eval_ondist.py experiment/awm_eval_tasks.json

In [ ]:
import subprocess, time, urllib.request, os, signal

VLLM = None
def serve(model, name, max_model_len=24576, gpu_util=0.90):
    """Start vLLM serving `model` as `name` with the tool + reasoning parsers, wait for health."""
    global VLLM
    stop()
    cmd = [
        "vllm", "serve", model, "--served-model-name", name,
        "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
        "--reasoning-parser", "deepseek_r1",
        "--max-model-len", str(max_model_len),
        "--gpu-memory-utilization", str(gpu_util),
        "--port", "8000",
    ]
    print("launching:", " ".join(cmd))
    VLLM = subprocess.Popen(cmd)
    for _ in range(600):  # up to ~10 min for download+load
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=2)
            print(f"\n{name} is up"); return
        except Exception:
            if VLLM.poll() is not None:
                raise RuntimeError("vLLM exited early — check the log above")
            time.sleep(2)
    raise TimeoutError("vLLM did not become healthy in time")

def stop():
    global VLLM
    if VLLM and VLLM.poll() is None:
        VLLM.send_signal(signal.SIGINT)
        try: VLLM.wait(timeout=60)
        except Exception: VLLM.kill()
    VLLM = None
    time.sleep(5)  # let the GPU free

## Finetuned model

In [ ]:
serve("Jakemu/Qwen3-4B-Thinking-awm-async-grpo-100", "ft")
!python experiment/awm_eval_ondist.py --model ft --output ft.json

## Base model

In [ ]:
serve("Qwen/Qwen3-4B-Thinking-2507", "base")
!python experiment/awm_eval_ondist.py --model base --output base.json
stop()

## Compare

In [ ]:
import json
b = json.load(open("base.json")); f = json.load(open("ft.json"))
bs, fs = b["summary"], f["summary"]
print(f"base     success_rate {bs['success_rate']:.3f}  ({bs['successes']}/{bs['num_tasks']})")
print(f"finetune success_rate {fs['success_rate']:.3f}  ({fs['successes']}/{fs['num_tasks']})")
print(f"delta    {fs['success_rate']-bs['success_rate']:+.3f}\n")

# Per-task flips (same fixed task set for both).
bm = {(t['scenario'],t['task_idx']): t['success'] for t in b['tasks']}
fm = {(t['scenario'],t['task_idx']): t['success'] for t in f['tasks']}
gained = [k for k in bm if fm.get(k) and not bm[k]]
lost   = [k for k in bm if bm[k] and not fm.get(k)]
print(f"gained (base✗ → ft✓): {len(gained)}")
for k in gained: print('   +', k[0], k[1])
print(f"lost   (base✓ → ft✗): {len(lost)}")
for k in lost: print('   -', k[0], k[1])